# Curriculum 02 · Lab 1 — Local Embeddings: BGE vs E5

**Goal:** Run two popular open-source embedders head to head on the same real
corpus and inspect the three numbers that decide everything downstream:
dimension, vector norm, and cosine similarity. An embedding turns text into a
vector of numbers, and the whole RAG pipeline stands on the assumption that
"similar meaning => similar vector" — this lab makes that assumption
measurable before any retrieval code runs.

```
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU) vs E5 (intfloat/multilingual-e5-base)
Metric      : embedding dimension, vector norm, cosine-similarity top-3 retrieval
Models      : BGEEmbedding (embeddings/bge.py) + E5Embedding (embeddings/e5.py)
Data        : rag-mini-wikipedia (first 20 passages, test questions 1606/1610/1604)
```

**Why local models:** no API keys, no per-token cost, no data leaving your
machine. BGE is an English retrieval model trained with normalized embeddings;
E5 covers many languages and is trained with instruction prefixes — E5 queries
are prefixed "query: " and passages "passage: " before embedding, which
`src/embeddings/e5.py` applies automatically. Prefix mismatches are a classic
silent retrieval killer.

This is the first lab of track 02-embeddings (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Imports every dependency (numpy, pandas, scikit-learn's cosine
similarity) plus the repo's two local embedders, and puts the repo-root
component library on `sys.path` so this notebook reuses `src/embeddings/bge.py`
and `src/embeddings/e5.py` exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE and E5 via
sentence-transformers — no API embeddings anywhere. `src/embeddings/bge.py`
builds the model with the current universal `HuggingFaceEmbeddings` class and
normalizes explicitly (`encode_kwargs` `normalize_embeddings=True`), while
`src/embeddings/e5.py` applies the `"query: "` / `"passage: "` instruction
prefixes automatically — same contract, two very different models.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script,
`python src/curriculum/02-embeddings/01-local-bge-e5.py`) or from the notebook's
own folder (the Jupyter default) — and `cd`s into it so every path stays
repo-relative.

**WHAT TO EXPECT:** no output — just a clean import. The models themselves are
loaded lazily when the comparison cell first calls them.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE + E5 embeddings
#   pandas                -> reads the passages/test.parquet corpus
#   scikit-learn          -> cosine_similarity for the top-k ranking
%pip install sentence-transformers pandas scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from embeddings.bge import BGEEmbedding  # noqa: E402
from embeddings.e5 import E5Embedding  # noqa: E402

## 1 · Configuration — the comparison's knobs

**WHAT:** The module-level constants that define the comparison: which corpus
files to read, how many passages to embed, which test questions to ask, how
many hits to print per question, and the preview width.

**WHY:** These are the knobs you tweak to re-run the experiment. `N_PASSAGES
= 20` takes the deterministic head of the 3200-passage corpus so CPU embedding
time stays low; `QUESTION_IDS = [1606, 1610, 1604]` are real rows of
`test.parquet` picked to make the retrieval section interesting; `TOP_K = 3`
is the answer horizon the lab inspects.


In [3]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the comparison
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 20  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, picked to match
TOP_K = 3
PREVIEW = 62  # max characters of passage text shown next to each hit
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"

## 2 · Load — corpus + questions from the fresh parquet files

**WHAT:** Two small readers: `load_passages` pulls the first `n` passages
(text + ids) from `passages.parquet`; `load_questions` pulls specific rows by
id from `test.parquet` and returns `(question_id, question_text)` pairs.

**WHY:** The corpus lives in the repo's fresh rag-mini-wikipedia set
(`Data/corpus/rag-mini-wikipedia/`), manifest-verified and fully local — no
downloads, no randomness. The ids let the retrieval section label every hit
back to a specific passage, which is how we see *where* each model finds an
answer.


In [4]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]

## 3 · Embed & compare — helpers shared by both models

**WHAT:** `run_model` embeds all passages in one batched call and each query
one at a time (`embed_query`) — exactly how real RAG behaves, where every
incoming question is embedded individually. `l2_norm` measures vector length,
`top_k_results` ranks passages by cosine similarity for a query and returns
the top-k `(id, score)` pairs, and `preview` flattens a passage for one-line
printing.

**WHY:** Keeping these model-agnostic is the point of the lab — the two
models differ only in construction and normalization, and every helper here
treats them identically. Cosine similarity is computed over numpy arrays, so
the comparison is fair for both unit-norm and unnormalized vectors.


In [5]:
# --------------------------------------------------------------------------
# 3. Embed & compare — helpers shared by both models
# --------------------------------------------------------------------------
def run_model(
    model: object, passages: list[str], questions: list[str]
) -> tuple[list[list[float]], list[list[float]]]:
    """Embed all passages and all questions with one model.

    Returns (passage_vectors, query_vectors). Passages are embedded in one
    batched call; queries one at a time (``embed_query``) because in real RAG
    each incoming question is embedded individually.
    """
    passage_vecs = model.embed_documents(passages)
    query_vecs = [model.embed_query(q) for q in questions]
    return passage_vecs, query_vecs


def l2_norm(vector: list[float]) -> float:
    """Euclidean length of an embedding vector."""
    return float(np.linalg.norm(np.asarray(vector, dtype=np.float32)))


def top_k_results(
    query_vec: list[float],
    passage_vecs: list[list[float]],
    passage_ids: list[int],
    k: int,
) -> list[tuple[int, float]]:
    """Rank passages by cosine similarity to the query; return top-k (id, score)."""
    matrix = np.asarray(passage_vecs, dtype=np.float32)
    sims = cosine_similarity(np.asarray([query_vec], dtype=np.float32), matrix)[0]
    order = np.argsort(sims)[::-1][:k]
    return [(passage_ids[i], float(sims[i])) for i in order]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")

## 4 · Run — load the corpus subset and the test questions

**WHAT:** The first step of the experiment: load the 20-passage subset and
the 3 test questions, then print what we are comparing.

**WHY:** The header printout makes the run self-describing — same corpus,
same questions, both models. Everything after this cell is measured on
exactly these inputs.


In [6]:
# --- 2. Load ---------------------------------------------------------
passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
questions = load_questions(TEST_PATH, QUESTION_IDS)

print("=" * 66)
print("Lab 01 — local embeddings: BGE vs E5 on rag-mini-wikipedia")
print(f"{BGE_MODEL_NAME}  vs  intfloat/multilingual-e5-base")
print("=" * 66)

print(f"\n[1] Corpus (deterministic subset, no randomness):")
print(f"    {len(passage_texts)} passages (first {N_PASSAGES} of 3200, ids {passage_ids[0]}..{passage_ids[-1]})")
print(f"    {len(questions)} questions from test.parquet:")
for qid, qtext in questions:
    print(f"      [{qid}] {qtext}")

Lab 01 — local embeddings: BGE vs E5 on rag-mini-wikipedia
BAAI/bge-base-en-v1.5  vs  intfloat/multilingual-e5-base

[1] Corpus (deterministic subset, no randomness):
    20 passages (first 20 of 3200, ids 0..19)
    3 questions from test.parquet:
      [1606] Is Uruguay's capital Montevideo?
      [1610] Who founded Montevideo?
      [1604] Is Uruguay located in the northwesten part of Africa?


## 5 · Embed — both models, dimension & norm

**WHAT:** Build one `BGEEmbedding` and one `E5Embedding`, embed every passage
and every question with each, and print the two numbers that define the
vector space: dimension and norm.

**WHY:** Both models produce 768-dim vectors and both come out unit-norm —
but through different mechanisms. BGE is normalized explicitly by
`src/embeddings/bge.py` (`encode_kwargs` `normalize_embeddings=True`); E5 ships
its own `2_Normalize` layer on the Hub, so it is unit-norm by construction.
A unit-norm vector makes cosine similarity identical to a dot product, which
matters when your vector database offers fast dot-product scoring — and a
surprising number of embedders do NOT normalize, so always check.


In [7]:
# --- 3. Embed with both models ---------------------------------------
models = {"BGE": BGEEmbedding(model_name=BGE_MODEL_NAME), "E5": E5Embedding()}
question_texts = [qtext for _, qtext in questions]
embedded = {
    name: run_model(model, passage_texts, question_texts)
    for name, model in models.items()
}

print("\n[2] Embedding vectors — dimension and norm:")
print(f"    {'model':<6}{'dim':>6}{'passage norm':>14}{'query norm':>14}")
for name, (pvecs, qvecs) in embedded.items():
    dim = len(pvecs[0])
    p_norm = l2_norm(pvecs[0])
    q_norm = l2_norm(qvecs[0])
    print(f"    {name:<6}{dim:>6}{p_norm:>14.4f}{q_norm:>14.4f}")
print("    BGE is normalized explicitly (encode_kwargs); E5 unit-norm via its")
print("    model's own 2_Normalize layer — cosine == dot product for both.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[2] Embedding vectors — dimension and norm:
    model    dim  passage norm    query norm
    BGE      768        1.0000        1.0000
    E5       768        1.0000        1.0000
    BGE is normalized explicitly (encode_kwargs); E5 unit-norm via its
    model's own 2_Normalize layer — cosine == dot product for both.


## 6 · Retrieve — top-3 per question, cosine similarity

**WHAT:** For each of the three test questions, rank the 20 passages by
cosine similarity for both models and print the top-3 hits with a preview of
the passage text.

**WHY:** This is the retrieval section — the same question, two vector
spaces. The scores are only comparable *within* one model's vector space:
a BGE 0.45 is not an E5 0.45. What matters is the ordering — which passage
ends up on top, and whether the two models agree.


In [8]:
# --- 4. Top-k retrieval per question ---------------------------------
print(f"\n[3] Top-{TOP_K} retrieval per question (cosine similarity):")
for i, (qid, qtext) in enumerate(questions):
    print(f'\n    Q[{qid}] "{qtext}"')
    for name in models:
        pvecs, qvecs = embedded[name]
        hits = top_k_results(qvecs[i], pvecs, passage_ids, TOP_K)
        print(f"      {name:<4} " + "  ".join(
            f"id {pid} {score:.4f}" for pid, score in hits
        ))
        for pid, score in hits:
            idx = passage_ids.index(pid)
            print(f"            {score:.4f}  {preview(passage_texts[idx])}")


[3] Top-3 retrieval per question (cosine similarity):

    Q[1606] "Is Uruguay's capital Montevideo?"
      BGE  id 15 0.7535  id 2 0.7206  id 0 0.7018
            0.7535  Uruguay's capital, Montevideo, was founded by the Spanish in t...
            0.7206  Montevideo was founded by the Spanish in the early 18th centur...
            0.7018  Uruguay (official full name in  ; pron.  , Eastern Republic of...
      E5   id 0 0.8789  id 2 0.8776  id 15 0.8736
            0.8789  Uruguay (official full name in  ; pron.  , Eastern Republic of...
            0.8776  Montevideo was founded by the Spanish in the early 18th centur...
            0.8736  Uruguay's capital, Montevideo, was founded by the Spanish in t...

    Q[1610] "Who founded Montevideo?"
      BGE  id 15 0.7872  id 2 0.7705  id 12 0.6685
            0.7872  Uruguay's capital, Montevideo, was founded by the Spanish in t...
            0.7705  Montevideo was founded by the Spanish in the early 18th centur...
            0.6685 

## 7 · Takeaway — what the vectors teach

**WHAT:** Read the numbers from the two tables and print the lesson the lab
teaches.

**WHY:** Same dimension (768) and both unit-norm — yet not interchangeable.
E5's wrapper adds `"query: "` / `"passage: "` prefixes that BGE never sees,
and BGE's normalization is explicit while E5's is baked into the model. When
you swap embedders, check the vector norm and the prefix handling together —
and remember retrieval scores are only comparable within one model's vector
space, never across models.


In [9]:
# --- 5. Takeaway -----------------------------------------------------
print("\n[4] Takeaway")
print("    Same dimension (768) and both unit-norm, yet not interchangeable:")
print("    E5's wrapper adds 'query: '/'passage: ' prefixes that BGE never")
print("    sees, and BGE's normalization is explicit while E5's is baked")
print("    into the model. When you swap embedders, check the vector norm")
print("    and the prefix handling together — and remember retrieval scores")
print("    are only comparable within one model's vector space, never across")
print("    models.")


[4] Takeaway
    Same dimension (768) and both unit-norm, yet not interchangeable:
    E5's wrapper adds 'query: '/'passage: ' prefixes that BGE never
    sees, and BGE's normalization is explicit while E5's is baked
    into the model. When you swap embedders, check the vector norm
    and the prefix handling together — and remember retrieval scores
    are only comparable within one model's vector space, never across
    models.
